In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os
data_folder = "/content/drive/MyDrive/veri-madenciligi-projesi/notebooks"  # <- burayı kendi Drive klasörüne göre değiştir

# Klasördeki csv dosyalarını listele
files = [f for f in os.listdir(data_folder) if f.lower().endswith('.csv')]
print("Found CSV files:", len(files))
for f in files:
    print("-", f)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found CSV files: 0


In [7]:
import pandas as pd
import os

folder = "/content/drive/MyDrive/veri-madenciligi-projesi/notebooks"

for file in os.listdir(folder):
    if file.endswith(".csv"):
        path = os.path.join(folder, file)

        print("Temizleniyor:", file)

        # Dosyayı oku
        raw = pd.read_csv(path, header=None)

        # 1) "Date" satırını bul
        header_row = raw[raw.apply(lambda row: row.str.contains("Date", na=False)).any(axis=1)]

        if len(header_row) == 0:
            print("Başlık bulunamadı, atlandı:", file)
            continue

        header_index = header_row.index[0]

        # 2) Bu satırı header olarak ayarla
        new_header = raw.iloc[header_index]

        # 3) Header'dan sonraki gerçek verileri al
        df = raw[(header_index+1):]
        df.columns = new_header
        df.reset_index(drop=True, inplace=True)

        # 4) Gereksiz "NaN" satırları temizle
        df = df.dropna(subset=["Date"])

        # 5) Temiz dosyayı tekrar kaydet
        df.to_csv(path, index=False)

        print("Temizlendi:", file)

print("Tüm dosyalar temizlendi")


Tüm dosyalar temizlendi


In [8]:
import pandas as pd
import os

folder = "/content/drive/MyDrive/veri-madenciligi-projesi/notebooks"

# Yahoo Finance dosyaları
broken_files = [
    "ZQ=F.csv", "IVV.csv", "GC=F.csv", "EURUSD=X.csv",
    "DXY.csv", "CPI.csv", "BTC-USD.csv", "^VIX.csv", "^GSPC.csv"
]

# Doğru Yahoo kolonları
correct_cols = ["Date","Close","High","Low","Open","Volume"]

for file in broken_files:
    path = os.path.join(folder, file)

    if not os.path.exists(path):
        print("Dosya yok:", file)
        continue

    print("Düzeltiliyor:", file)

    df = pd.read_csv(path, header=None)

    # Eğer Date ilk satırdaysa ve sütun adları yoksa
    if df.iloc[0].str.contains("Date", na=False).any():
        df = df[1:]  # Date satırını at
        df.columns = correct_cols[:len(df.columns)]  # mümkün olan kolonları ekle

        df.to_csv(path, index=False)
        print("Düzeltildi:", file)
    else:
        print("Zaten düzgün olabilir:", file)

print("Tüm bozuk dosyalar düzeltildi!")


Dosya yok: ZQ=F.csv
Dosya yok: IVV.csv
Dosya yok: GC=F.csv
Dosya yok: EURUSD=X.csv
Dosya yok: DXY.csv
Dosya yok: CPI.csv
Dosya yok: BTC-USD.csv
Dosya yok: ^VIX.csv
Dosya yok: ^GSPC.csv
Tüm bozuk dosyalar düzeltildi!


In [9]:
#Date+Value seçen fonksiyon
import os
import pandas as pd
import numpy as np

data_folder = "/content/drive/MyDrive/veri-madenciligi-projesi/notebooks"


# listele
files = [f for f in os.listdir(data_folder) if f.lower().endswith('.csv')]
print("Found", len(files), "csv files.\n")

def load_and_select(path):
    df = pd.read_csv(path, dtype=str)

    df.columns = [c.strip() for c in df.columns]

    # Tarih sütunu bul
    date_candidates = ['Date','DATE','date','observation_date','date_time','timestamp','index']
    date_col = next((c for c in date_candidates if c in df.columns), None)

    # Eğer doğrudan yoksa header'ı farklı satırlarda arayalım (0-5 arası)
    if date_col is None:
        found = False
        for h in range(0,6):
            try:
                tmp = pd.read_csv(path, header=h, nrows=5, dtype=str)
                tmp.columns = [c.strip() for c in tmp.columns]
                if any([c for c in tmp.columns if c.lower().strip()=='date' or 'date' in c.lower()]):
                    date_col = [c for c in tmp.columns if 'date' in c.lower()][0]
                    df = pd.read_csv(path, header=h, dtype=str)
                    df.columns = [c.strip() for c in df.columns]
                    found = True
                    break
            except Exception:
                continue
        if not found and date_col is None:

            for c in df.columns[:3]:
                try:
                    pd.to_datetime(df[c].iloc[:5])
                    date_col = c
                    break
                except Exception:
                    pass

    if date_col is None:
        raise ValueError(f"No date-like column found in {os.path.basename(path)}")

    # Değer sütunu
    value_candidates = ['Adj Close','Adj_Close','AdjClose','Close','close','Value','VALUE','value']
    value_col = next((c for c in value_candidates if c in df.columns), None)

    if value_col is None:
        #sayısal tip dönüşümü
        for c in df.columns:
            try:
                _ = pd.to_numeric(df[c].dropna().iloc[:5])
                value_col = c
            except Exception:
                continue

    if value_col is None:
        raise ValueError(f"No numeric value column found in {os.path.basename(path)}")

    #  Sütün seçimi
    out = df[[date_col, value_col]].copy()
    out.columns = ['Date','Value']
    # parse date
    out['Date'] = pd.to_datetime(out['Date'], errors='coerce')
    out = out.dropna(subset=['Date'])
    # try convert Value to numeric.....
    out['Value'] = out['Value'].astype(str).str.replace(',','').str.strip()
    out['Value'] = pd.to_numeric(out['Value'], errors='coerce')
    # drop rows where Value is NaN
    out = out.dropna(subset=['Value'])
    # drop duplicates
    out = out.drop_duplicates(subset=['Date'], keep='last').sort_values('Date').reset_index(drop=True)
    return out

# quick test: load all & show heads
sample = {}
for f in files:
    p = os.path.join(data_folder, f)
    try:
        df = load_and_select(p)
        sample[f] = df.head(3)
    except Exception as e:
        sample[f] = f"ERROR: {e}"

# print results
for k,v in sample.items():
    print("==", k, "==")
    print(v)
    print()


Found 0 csv files.



In [10]:
# read & merge all CSVs into one DataFrame
merged = None
for f in files:
    p = os.path.join(data_folder, f)
    try:
        df = load_and_select(p)
    except Exception as e:
        print("SKIP", f, "->", e)
        continue
    colname = os.path.splitext(f)[0]  # file name without .csv
    df = df.rename(columns={'Value': colname})
    if merged is None:
        merged = df
    else:
        merged = merged.merge(df, on='Date', how='outer')

# Check if merged DataFrame was created
if merged is None:
    print("Error: No CSV files were processed to create the merged DataFrame. 'files' list might be empty.")
else:
    # sort and reset
    merged = merged.sort_values('Date').reset_index(drop=True)
    print("Merged shape:", merged.shape)
    print(merged.head())

Error: No CSV files were processed to create the merged DataFrame. 'files' list might be empty.


In [11]:
# === Logistic Regression (Regression mode) ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score

# Hedef değişkenin adını kendine göre düzenle
target = "Value"   # örnek
X = df.drop(columns=[target])
y = df[target]

# Veriyi ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ölçekleme
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model
log_reg = LogisticRegression(max_iter=5000)

# Eğitim
log_reg.fit(X_train_scaled, y_train)

# Tahmin
y_pred_lr = log_reg.predict(X_test_scaled)

# Sonuçlar
print("=== Logistic Regression Results ===")
print("MSE :", mean_squared_error(y_test, y_pred_lr))
print("R²  :", r2_score(y_test, y_pred_lr))


NameError: name 'df' is not defined

In [ ]:
# === KNN Regression ===
from sklearn.neighbors import KNeighborsRegressor

# Model
knn = KNeighborsRegressor(n_neighbors=5)

# Eğitim
knn.fit(X_train_scaled, y_train)

# Tahmin
y_pred_knn = knn.predict(X_test_scaled)

# Sonuçlar
print("=== KNN Regression Results ===")
print("MSE :", mean_squared_error(y_test, y_pred_knn))
print("R²  :", r2_score(y_test, y_pred_knn))


In [ ]:
# Temel eksik doldurma
merged_filled = merged.copy()
merged_filled = merged_filled.set_index('Date')

# forward fill then back fill
merged_filled = merged_filled.ffill().bfill()

# Eğer bazı sütunlar quarter/ monthly olduğundan çok boşsa, uyarır
na_counts = merged.isna().sum()
print("Na counts before fill:\n", na_counts[na_counts>0])

print("After fill, any Na left?", merged_filled.isna().any().any())

# reset index
merged_final = merged_filled.reset_index()

# kaydet
out_path = os.path.join(data_folder, "merged_data.csv")
merged_final.to_csv(out_path, index=False)
print("Dosya kaydedildi:", out_path)
merged_final.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# eksik değerlere bak
print("Columns and non-null counts:")
print(merged_final.count())

# numeric only for correlation
num = merged_final.select_dtypes(include=[np.number]).copy()
corr = num.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix (merged features)")
plt.show()


In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
# Merged dosyanı oku
merged_final = pd.read_csv("/content/drive/MyDrive/veri-madenciligi-projesi/notebooks/merged_data.csv")

# Tarih kolonunu düzeltme
merged_final['Date'] = pd.to_datetime(merged_final['Date'])
merged_final = merged_final.sort_values('Date')
merged_final = merged_final.set_index('Date')

# Eksik değerler
print("Eksik değerler (önce):")
print(merged_final.isnull().sum())

# Zaman serisi interpolasyonu
merged_final = merged_final.interpolate(method='linear')

# Baştaki ve sondaki boş değerler
merged_final = merged_final.ffill().bfill()

# Final kontrol
print("Eksik değerler (sonra):")
print(merged_final.isnull().sum())

# Temiz versiyon save
merged_final.to_csv("/content/drive/MyDrive/veri-madenciligi-projesi/merged_final_clean.csv")


In [ ]:
import pandas as pd
import numpy as np

#  Dosyayı oku
path = "/content/drive/MyDrive/veri-madenciligi-projesi/merged_final_clean.csv"
df = pd.read_csv(path)

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Target kolonları
df["Target_1d"] = df["BTC-USD"].shift(-1)
df["Target_7d"] = df["BTC-USD"].shift(-7)
df["Target_30d"] = df["BTC-USD"].shift(-30)
df["Target_365d"] = df["BTC-USD"].shift(-365)

#Lag'lerin sayısı
lags = [1, 3, 7, 30, 90, 365]

#Tüm kolonlara gecikme ekleme
cols = [c for c in df.columns if c not in ["Date", "Target_1d", "Target_7d", "Target_30d", "Target_365d"]]

for col in cols:
    for lag in lags:
        df[f"{col}_lag{lag}"] = df[col].shift(lag)

# Son satırlardaki NaN'ların temizlenmesi
df = df.dropna().reset_index(drop=True)

# Yeni dataset save
output_path = "/content/drive/MyDrive/veri-madenciligi-projesi/merged_with_lags.csv"
df.to_csv(output_path, index=False)

print("Yeni dosya kaydedildi:")
print(output_path)
print("\nÖrnek satırlar:")
print(df.head())
